In [1]:
from pathlib import Path
import json

import joblib
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [2]:
# Пути

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
EXPERIMENTS_DIR = PROJECT_ROOT / "experiments"

LOGISTIC_TUNED_DIR = EXPERIMENTS_DIR / "logistic_regression_tuned"
TREE_TUNED_DIR = EXPERIMENTS_DIR / "decision_tree_tuned"

LOGISTIC_TUNED_DIR.mkdir(parents=True, exist_ok=True)
TREE_TUNED_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = PROCESSED_DATA_DIR / "train.csv"
VALID_PATH = PROCESSED_DATA_DIR / "valid.csv"

print("Train path:", TRAIN_PATH)
print("Valid path:", VALID_PATH)
print("Logistic tuned dir:", LOGISTIC_TUNED_DIR)
print("Decision tree tuned dir:", TREE_TUNED_DIR)

Train path: c:\Users\Qekqq\Desktop\Devops\devops_hw_1\data\processed\train.csv
Valid path: c:\Users\Qekqq\Desktop\Devops\devops_hw_1\data\processed\valid.csv
Logistic tuned dir: c:\Users\Qekqq\Desktop\Devops\devops_hw_1\experiments\logistic_regression_tuned
Decision tree tuned dir: c:\Users\Qekqq\Desktop\Devops\devops_hw_1\experiments\decision_tree_tuned


In [3]:
# Загрузка данных

train_data = pd.read_csv(TRAIN_PATH)
valid_data = pd.read_csv(VALID_PATH)

TARGET_COLUMN = "Outcome"

X_train = train_data.drop(columns=[TARGET_COLUMN])
y_train = train_data[TARGET_COLUMN]

X_valid = valid_data.drop(columns=[TARGET_COLUMN])
y_valid = valid_data[TARGET_COLUMN]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_valid:", X_valid.shape)
print("y_valid:", y_valid.shape)

X_train: (536, 8)
y_train: (536,)
X_valid: (116, 8)
y_valid: (116,)


In [4]:
# Функция для оценки модели

def evaluate_model(model, X, y, model_name):
    y_pred = model.predict(X)

    metrics = {
        "model_name": model_name,
        "accuracy": float(accuracy_score(y, y_pred)),
        "precision": float(precision_score(y, y_pred)),
        "recall": float(recall_score(y, y_pred)),
        "f1": float(f1_score(y, y_pred)),
    }

    conf_matrix = confusion_matrix(y, y_pred)

    conf_matrix_df = pd.DataFrame(
        conf_matrix,
        index=["Actual 0", "Actual 1"],
        columns=["Predicted 0", "Predicted 1"]
    )

    return metrics, conf_matrix_df, y_pred

# Подбор гиперпараметров LogisticRegression

In [5]:
logistic_pipeline = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                solver="liblinear",
                random_state=57
            )
        )
    ]
)

logistic_param_grid = {
    "classifier__C": [0.01, 0.1, 1, 10, 100],
    "classifier__penalty": ["l1", "l2"],
    "classifier__class_weight": [None, "balanced"],
}

logistic_grid_search = GridSearchCV(
    estimator=logistic_pipeline,
    param_grid=logistic_param_grid,
    scoring="f1",
    cv=5,
    n_jobs=-1
)

logistic_grid_search.fit(X_train, y_train)

print("Best LogisticRegression params:")
print(logistic_grid_search.best_params_)

print("Best CV F1-score:")
print(logistic_grid_search.best_score_)

Best LogisticRegression params:
{'classifier__C': 1, 'classifier__class_weight': 'balanced', 'classifier__penalty': 'l1'}
Best CV F1-score:
0.6533604256951104


c:\Users\Qekqq\Desktop\Devops\devops_hw_1\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\Qekqq\Desktop\Devops\devops_hw_1\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


In [ ]:
# Оценка tuned LogisticRegression на validation

best_logistic_model = logistic_grid_search.best_estimator_

logistic_tuned_metrics, logistic_tuned_conf_matrix, logistic_valid_pred = evaluate_model(
    best_logistic_model,
    X_valid,
    y_valid,
    "LogisticRegressionTuned"
)

logistic_tuned_metrics["best_cv_f1"] = float(logistic_grid_search.best_score_)
logistic_tuned_metrics["best_params"] = logistic_grid_search.best_params_

display(pd.DataFrame([logistic_tuned_metrics]))
display(logistic_tuned_conf_matrix)

print(classification_report(y_valid, logistic_valid_pred))

,model_name,accuracy,precision,recall,f1,best_cv_f1,best_params
0,LogisticRegressionTuned,0.793103,0.673469,0.804878,0.733333,0.65336,"{'classifier__C': 1, 'classifier__class_weight..."


,Predicted 0,Predicted 1
Actual 0,59,16
Actual 1,8,33


              precision    recall  f1-score   support

           0       0.88      0.79      0.83        75
           1       0.67      0.80      0.73        41

    accuracy                           0.79       116
   macro avg       0.78      0.80      0.78       116
weighted avg       0.81      0.79      0.80       116



In [7]:
# Сохранение tuned LogisticRegression

LOGISTIC_TUNED_MODEL_PATH = LOGISTIC_TUNED_DIR / "model.joblib"
LOGISTIC_TUNED_METRICS_PATH = LOGISTIC_TUNED_DIR / "metrics.json"

joblib.dump(best_logistic_model, LOGISTIC_TUNED_MODEL_PATH)

with open(LOGISTIC_TUNED_METRICS_PATH, "w", encoding="utf-8") as file:
    json.dump(logistic_tuned_metrics, file, indent=4)

print("Tuned LogisticRegression model saved to:", LOGISTIC_TUNED_MODEL_PATH)
print("Tuned LogisticRegression metrics saved to:", LOGISTIC_TUNED_METRICS_PATH)

Tuned LogisticRegression model saved to: c:\Users\Qekqq\Desktop\Devops\devops_hw_1\experiments\logistic_regression_tuned\model.joblib
Tuned LogisticRegression metrics saved to: c:\Users\Qekqq\Desktop\Devops\devops_hw_1\experiments\logistic_regression_tuned\metrics.json


# Подбор гиперпараметров DecisionTreeClassifier

In [8]:
tree_model = DecisionTreeClassifier(random_state=57)

tree_param_grid = {
    "max_depth": [2, 3, 4, 5, 6, None],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 5, 10],
    "criterion": ["gini", "entropy"],
    "class_weight": [None, "balanced"],
}

tree_grid_search = GridSearchCV(
    estimator=tree_model,
    param_grid=tree_param_grid,
    scoring="f1",
    cv=5,
    n_jobs=-1
)

tree_grid_search.fit(X_train, y_train)

print("Best DecisionTreeClassifier params:")
print(tree_grid_search.best_params_)

print("Best CV F1-score:")
print(tree_grid_search.best_score_)

Best DecisionTreeClassifier params:
{'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 4, 'min_samples_leaf': 10, 'min_samples_split': 2}
Best CV F1-score:
0.6962628750229582


In [9]:
# Оценка tuned DecisionTree на validation

best_tree_model = tree_grid_search.best_estimator_

tree_tuned_metrics, tree_tuned_conf_matrix, tree_valid_pred = evaluate_model(
    best_tree_model,
    X_valid,
    y_valid,
    "DecisionTreeClassifierTuned"
)

tree_tuned_metrics["best_cv_f1"] = float(tree_grid_search.best_score_)
tree_tuned_metrics["best_params"] = tree_grid_search.best_params_

display(pd.DataFrame([tree_tuned_metrics]))
display(tree_tuned_conf_matrix)

print(classification_report(y_valid, tree_valid_pred))

,model_name,accuracy,precision,recall,f1,best_cv_f1,best_params
0,DecisionTreeClassifierTuned,0.767241,0.62069,0.878049,0.727273,0.696263,"{'class_weight': 'balanced', 'criterion': 'gin..."


,Predicted 0,Predicted 1
Actual 0,53,22
Actual 1,5,36


              precision    recall  f1-score   support

           0       0.91      0.71      0.80        75
           1       0.62      0.88      0.73        41

    accuracy                           0.77       116
   macro avg       0.77      0.79      0.76       116
weighted avg       0.81      0.77      0.77       116



In [10]:
# Сохранение tuned DecisionTree

TREE_TUNED_MODEL_PATH = TREE_TUNED_DIR / "model.joblib"
TREE_TUNED_METRICS_PATH = TREE_TUNED_DIR / "metrics.json"

joblib.dump(best_tree_model, TREE_TUNED_MODEL_PATH)

with open(TREE_TUNED_METRICS_PATH, "w", encoding="utf-8") as file:
    json.dump(tree_tuned_metrics, file, indent=4)

print("Tuned DecisionTreeClassifier model saved to:", TREE_TUNED_MODEL_PATH)
print("Tuned DecisionTreeClassifier metrics saved to:", TREE_TUNED_METRICS_PATH)

Tuned DecisionTreeClassifier model saved to: c:\Users\Qekqq\Desktop\Devops\devops_hw_1\experiments\decision_tree_tuned\model.joblib
Tuned DecisionTreeClassifier metrics saved to: c:\Users\Qekqq\Desktop\Devops\devops_hw_1\experiments\decision_tree_tuned\metrics.json


In [11]:
tuned_metrics_df = pd.DataFrame([
    logistic_tuned_metrics,
    tree_tuned_metrics
])

display(tuned_metrics_df[[
    "model_name",
    "accuracy",
    "precision",
    "recall",
    "f1",
    "best_cv_f1",
    "best_params"
]].sort_values(by="f1", ascending=False))

,model_name,accuracy,precision,recall,f1,best_cv_f1,best_params
0,LogisticRegressionTuned,0.793103,0.673469,0.804878,0.733333,0.653360,"{'classifier__C': 1, 'classifier__class_weight..."
1,DecisionTreeClassifierTuned,0.767241,0.620690,0.878049,0.727273,0.696263,"{'class_weight': 'balanced', 'criterion': 'gin..."
